<a href="https://colab.research.google.com/github/MZiaAfzal71/Melting-Point-Prediction-of-Boronic-Acids/blob/main/Data%20Files/Scripts%20and%20Models/Hyperparameter_Tunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Colab Notebook: Nested Cross-Validation for Boronic Acid Descriptor Evaluation

## 🧭 Overview (Place this at the very top of the file)

This notebook automates nested cross-validation experiments for evaluating the performance of XGBoost regression models on different molecular descriptor datasets derived from Boronic Acids.

Each descriptor type (e.g., Mordred, Morgan, Coulomb Matrix, Path-Weighted Atom Vector) is treated as a separate input representation of the same molecules.

The pipeline performs:

1.   🧩 Dataset Loading (from GitHub repository)

2.   🧠 Automatic Feature Reduction for high-dimensional descriptors (e.g., Mordred)

3.  ⚙️ Randomized Hyperparameter Search with Early Stopping

4.  🔁 Nested Cross-Validation (Outer + Inner folds)

5.  📊 Automatic Logging and Saving of Model Results

This helps determine which descriptor best predicts the Melting Point of Boronic Acids using robust and reproducible evaluation.

## 💾 Cell 1: Clone Repository and Set Working Directory

The following cell clones the GitHub repository “Melting-Point-Prediction-of-Boronic-Acids” and changes the current working directory to the folder containing the data files, scripts, and pre-trained models used in this project.

In [1]:
!git clone https://github.com/MZiaAfzal71/Melting-Point-Prediction-of-Boronic-Acids
%cd Melting-Point-Prediction-of-Boronic-Acids/Data\ Files/Scripts\ and\ Models

Cloning into 'Melting-Point-Prediction-of-Boronic-Acids'...
remote: Enumerating objects: 354, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 354 (delta 47), reused 2 (delta 2), pack-reused 254 (from 2)
Receiving objects: 100% (354/354), 24.80 MiB | 31.23 MiB/s, done.
Resolving deltas: 100% (104/104), done.
/content/Melting-Point-Prediction-of-Boronic-Acids/Data Files/Scripts and Models


## 📦 Cell 2: Import Dependencies

In [6]:
# Requirements: xgboost, scikit-learn, numpy, pandas, joblib
import os
import random
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from xgboost import XGBRegressor

## ⚙️ Cell 3: Configuration and Directory Setup

In [3]:
# ⚙️ Configuration and Directory Setup
# ----------------------------------------------------
# This cell defines important configuration variables used throughout the notebook.
# It ensures that results are reproducible, files are correctly located,
# and output directories exist before any model training begins.

# --------------------------------------------------------------------
# Config
# --------------------------------------------------------------------
RANDOM_STATE = 42                # Ensures reproducibility of random processes
np.random.seed(RANDOM_STATE)     # Sets global NumPy seed for consistency

# -------------------------
# User Configuration
# -------------------------
DATA_DIR = Path("Excel Files")   # Directory containing descriptor datasets (Excel format)
OUT_DIR = Path("../Results")     # Folder where model outputs and summaries will be saved
OUT_DIR.mkdir(exist_ok=True)     # Create the output directory if it doesn't exist

## ⚙️ Cell 4: Define Core Functions

* Data Loading
* Feature Reduction
* CV Evaluation
* Hyperparameter Search and Nested Cross-Validation Functions
* Nested CV Driver Function

In [9]:
# Filename convention: {descriptor}.csv
# excel file must contain feature columns + a column "Melting Point"
# If SMILES present, drop it before training
def load_xy(descriptor_name, data_dir=DATA_DIR, target_col="Melting Point"):
    fn = data_dir / f"{descriptor_name}.xlsx"
    if not fn.exists():
        raise FileNotFoundError(f"Expected file: {fn}")
    df = pd.read_excel(fn)
    df = df.fillna(0)
    # if target_col not in df.columns:
    #     raise ValueError(f"target column '{target_col}' not found in {fn}. Found columns: {df.columns.tolist()}")
    X = df.iloc[:, 3:].values   # descriptors columns starts from 10th column onward
    y = df[target_col].values   # Melting Point column contains the target property
    return X, y, df


def reduce_features(X, y, max_features=None, var_thresh=1e-5):
    """
    Reduce the number of features for faster model training.

    Steps:
    1. Remove near-constant (low-variance) features.
    2. Select the top `max_features` features most correlated with y.

    Parameters:
        X (np.ndarray): Input feature matrix.
        y (np.ndarray): Target vector.
        max_features (int, optional): Max number of features to retain.
                                      Defaults to number of samples.
        var_thresh (float): Minimum variance to keep a feature.

    Returns:
        np.ndarray: Reduced feature matrix.
        np.ndarray: Indices of selected columns.
    """
    n_samples, n_features = X.shape
    if max_features is None:
        max_features = min(n_samples, n_features)

    # Step 1: Remove nearly constant columns
    selector = VarianceThreshold(threshold=var_thresh)
    X_reduced = selector.fit_transform(X)
    print(f"After variance filtering: {X_reduced.shape[1]} features remain")

    # Step 2: Select top correlated features (if still too many)
    if X_reduced.shape[1] > max_features:
        selector2 = SelectKBest(score_func=f_regression, k=max_features)
        X_final = selector2.fit_transform(X_reduced, y)
        selected_idx = selector2.get_support(indices=True)
        print(f"After feature selection: {X_final.shape[1]} features remain")
        return X_final, selected_idx
    else:
        return X_reduced, np.arange(X_reduced.shape[1])


# Param space for random search (XGBoost constructor arguments)
param_dist_xgb = {
    "n_estimators": [200, 400, 800],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0.0,  1.0],
    "reg_lambda": [1.0, 3.0],
    # you can add more if you like
}

def sample_params(param_dist):
    return {k: random.choice(v) for k, v in param_dist.items()}

# --------------------------------------------------------------------
# Inner CV evaluator that uses early stopping
# --------------------------------------------------------------------
def evaluate_params_with_inner_cv(X, y, params,
                                  inner_splits=3, random_state=RANDOM_STATE,
                                  early_stopping_rounds=30):
    """
    For one hyperparameter configuration, perform inner k-fold CV with early stopping.
    Returns the mean validation MAE across inner folds.
    """
    kf = KFold(n_splits=inner_splits, shuffle=True, random_state=random_state)
    maes = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y), start=1):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        # ensure eval_metric is provided in constructor (avoid passing to fit)
        model = XGBRegressor(
            objective="reg:squarederror",
            n_jobs=-1,
            random_state=int(random_state + fold),
            eval_metric="mae",
            early_stopping_rounds=early_stopping_rounds,
            **params
        )

        # fit with early stopping monitoring the validation fold (MAE)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        y_val_pred = model.predict(X_val)
        maes.append(mean_absolute_error(y_val, y_val_pred))

    return float(np.mean(maes)), float(np.std(maes))

# --------------------------------------------------------------------
# Outer loop: nested CV with random search + early stopping
# --------------------------------------------------------------------
def nested_cv_xgb_earlystop(X, y,
                            outer_splits=5, inner_splits=3,
                            n_iter_search=15, random_state=RANDOM_STATE,
                            out_prefix="exp"):
    """
    Runs nested CV:
      - outer KFold for evaluation
      - inner KFold for hyperparam search (random sampling) with early stopping
    Returns:
      - df_folds: per-fold metrics and params
      - summary: aggregated metrics
    """
    outer_cv = KFold(n_splits=outer_splits, shuffle=True, random_state=random_state)
    results = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        print(f"\n--- Outer fold {fold_idx}/{outer_splits} ---")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Random search over param_dist_xgb
        best_score = float("inf")
        best_params = None
        best_std = None

        for i in range(n_iter_search):
            candidate = sample_params(param_dist_xgb)
            mean_mae, std_mae = evaluate_params_with_inner_cv(
                X_train, y_train, params=candidate,
                inner_splits=inner_splits,
                random_state=random_state + i,
                early_stopping_rounds=30
            )
            # lower MAE is better
            if mean_mae < best_score:
                best_score = mean_mae
                best_params = candidate
                best_std = std_mae

            if (i+1) % max(1, n_iter_search//5) == 0:
                print(f"  Tried {i+1}/{n_iter_search} candidates, best MAE so far: {best_score:.4f}")

        print(f"> Best inner MAE (avg) for fold {fold_idx}: {best_score:.4f} ± {best_std:.4f}")
        print("  Best params:", best_params)

        # Retrain best on the entire outer training set, with a small held-out validation for early stopping
        X_tr_final, X_val_final, y_tr_final, y_val_final = train_test_split(
            X_train, y_train, test_size=0.15, random_state=random_state
        )

        best_model = XGBRegressor(
            objective="reg:squarederror",
            n_jobs=-1,
            random_state=int(random_state + fold_idx),
            eval_metric="mae",
            early_stopping_rounds=30,
            **best_params
        )

        best_model.fit(
            X_tr_final, y_tr_final,
            eval_set=[(X_val_final, y_val_final)],
            verbose=False
        )

        y_test_pred = best_model.predict(X_test)
        mae = mean_absolute_error(y_test, y_test_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
        r2 = r2_score(y_test, y_test_pred)

        # feature importances (aligned with columns in X)
        try:
            fi = best_model.feature_importances_.tolist()
        except Exception:
            fi = None

        # Save scaler + model together
        joblib.dump({"model": best_model},
                    OUT_DIR / f"{out_prefix}_fold{fold_idx}_pipeline.joblib")

        results.append({
            "fold": fold_idx,
            "mae": float(mae),
            "rmse": float(rmse),
            "r2": float(r2),
            "best_inner_mae": float(best_score),
            "best_inner_mae_std": float(best_std),
            "best_params": best_params,
            "feature_importances": fi
        })

    df_res = pd.DataFrame(results)
    summary = {
        "mae_mean": df_res["mae"].mean(),
        "mae_std": df_res["mae"].std(),
        "rmse_mean": df_res["rmse"].mean(),
        "r2_mean": df_res["r2"].mean()
    }
    return df_res, summary



## 🚀 Cell 5: Run Nested Cross-Validation on Multiple Descriptors

# ⚠️ NOTE:
* This cell performs a full nested cross-validation with random search and early stopping,
* which is computationally intensive. Google Colab provides only 2 CPU cores by default,resulting in significantly slower execution.

* For faster performance, it is recommended to run this notebook on Kaggle (or any better platform),
* which offers 4 CPU cores by default and executes this workflow considerably faster.

* The results reported in this study were obtained using the Kaggle environment.

In [10]:
# -------------------------
# Sweep over all descriptors
# -------------------------

descriptors = ["Boronic_MACCS_fingerprint", "Boronic_Morgan_fingerprint",
               "CoulombMatrix_BoronicAcids_Desc", "Boronic_Mordred_3DC",
               "Boronic_Bonds_Desc_Boron_En"]

all_experiments = []
for desc in descriptors:
    print(f">>> Running nested CV for {desc}")
    try:
        X, y, raw_df = load_xy(desc)
        if X.shape[1] > len(raw_df):
          # Reduce Mordred descriptors to speed up XGBoost tuning
          X_reduced, selected_idx = reduce_features(X, y)
        else:
          X_reduced = X


        print(f"Final X shape: {X_reduced.shape}")
    except Exception as e:
        print(f"Skipping {desc} because: {e}")
        continue

    out_prefix = f"{desc}"
    per_fold_df, per_exp_summary = nested_cv_xgb_earlystop(X_reduced, y,
                        outer_splits=5, inner_splits=3,
                        n_iter_search=50, random_state=RANDOM_STATE,
                        out_prefix=out_prefix)

    # save outputs
    per_fold_df.to_csv(OUT_DIR / f"{out_prefix}_cv_folds.csv", index=False)
    with open(OUT_DIR / f"{out_prefix}_summary.json", "w") as fh:
        json.dump(per_exp_summary, fh, indent=2)

    # append a flat record for later aggregation
    all_experiments.append({
        "Descriptor": desc,
        "mae_mean": per_exp_summary["mae_mean"],
        "mae_std": per_exp_summary["mae_std"],
        "rmse_mean": per_exp_summary["rmse_mean"],
        "r2_mean": per_exp_summary["r2_mean"]
    })

# Save global table
pd.DataFrame(all_experiments).to_csv(OUT_DIR / "all_xgb_results_summary.csv", index=False)
print("All experiments finished. Results saved to", OUT_DIR)

>>> Running nested CV for CoulombMatrix_BoronicAcids_Desc
After variance filtering: 4539 features remain
After feature selection: 605 features remain
Final X shape: (605, 605)

--- Outer fold 1/5 ---
  Tried 10/50 candidates, best MAE so far: 47.7801
  Tried 20/50 candidates, best MAE so far: 47.7801
  Tried 30/50 candidates, best MAE so far: 47.7801
  Tried 40/50 candidates, best MAE so far: 47.7801
  Tried 50/50 candidates, best MAE so far: 47.6118
> Best inner MAE (avg) for fold 1: 47.6118 ± 2.1517
  Best params: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 1.0, 'reg_alpha': 0.0, 'reg_lambda': 1.0}

--- Outer fold 2/5 ---
  Tried 10/50 candidates, best MAE so far: 46.9125
  Tried 20/50 candidates, best MAE so far: 46.0641
  Tried 30/50 candidates, best MAE so far: 46.0641
  Tried 40/50 candidates, best MAE so far: 45.3248
  Tried 50/50 candidates, best MAE so far: 45.3248
> Best inner MAE (avg) for fold 2: 45.3248 ± 3.5363
  Best 

KeyboardInterrupt: 